In [2]:
import tensorflow as tf
import sys

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")

# Test GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU devices found: {len(gpus)}")

if gpus:
    print("✅ Metal GPU acceleration available!")
    print(f"GPU details: {gpus[0]}")
    
    # Test a simple computation
    try:
        with tf.device('/GPU:0'):
            a = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
            c = tf.matmul(a, b)
            print("✅ GPU computation test successful!")
    except Exception as e:
        print(f"⚠️ GPU test failed: {e}")
else:
    print("⚠️ No GPU found, will use CPU")

# Memory info
print(f"Available memory: ~32GB (M1 Pro unified memory)")

Python version: 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
TensorFlow version: 2.15.0
GPU devices found: 1
✅ Metal GPU acceleration available!
GPU details: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
✅ GPU computation test successful!
Available memory: ~32GB (M1 Pro unified memory)


In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import argparse
import random
from pathlib import Path
from typing import Tuple, Dict, List, Optional
import cv2

warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV3Large

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import albumentations as A

## Create Model

In [9]:

config = {
            'image_size': (224, 224),
            'batch_size': 32,
            'num_classes': 5,
            'class_names': ['Healthy', 'Benign', 'OPMD', 'OCA', 'Unknown'],
            'epochs': 10,
            'learning_rate': 1e-4,
            'weight_decay': 1e-4,
            'dropout_rate': 0.3,
            'label_smoothing': 0.1,
            'mixed_precision': True,
            'data_augmentation_strength': 'medium',
        }

"""Create optimized model architecture for mobile deployment"""
inputs = keras.Input(shape=(config['image_size'], 3), name='input_image')

# Data augmentation layers (applied during training only)
# Use float32 for augmentation layers to avoid mixed precision issues
x = tf.cast(inputs, tf.float32)
x = layers.RandomFlip("horizontal")(x)
x = layers.RandomRotation(0.05)(x)
x = layers.RandomZoom(0.05)(x)
x = layers.RandomContrast(0.1)(x)

# Base model - MobileNetV3Large
base_model = MobileNetV3Large(
    input_shape=(224,224, 3),
    include_top=False,
    weights='imagenet',
    alpha=1.0,
    minimalistic=False,
    include_preprocessing=False,
)

# Fine-tuning strategy
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Add base model
x = base_model(x, training=True)

# Custom head for medical classification
x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
x = layers.BatchNormalization(name='bn1')(x)

# First dense layer
x = layers.Dense(512, name='dense1')(x)
x = layers.BatchNormalization(name='bn2')(x)
x = layers.Activation('relu', name='relu1')(x)
x = layers.Dropout(config['dropout_rate'], name='dropout1')(x)

# Second dense layer
x = layers.Dense(256, name='dense2')(x)
x = layers.BatchNormalization(name='bn3')(x)
x = layers.Activation('relu', name='relu2')(x)
x = layers.Dropout(config['dropout_rate'] * 0.7, name='dropout2')(x)

# Output layer - always use float32 for predictions to avoid precision issues
outputs = layers.Dense(config['num_classes'], activation='softmax', dtype='float32', name='predictions')(x)

model = keras.Model(inputs, outputs, name='OralCancerScreening')

TypeError: Dimension value must be integer or None or have an __index__ method, got value '(224, 224)' with type '<class 'tuple'>'

In [8]:
model